## Cleaning on the Datasets will be performed in this Notebook

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load Data

In [ ]:
# 2022-2023
hms_23 = pd.read_csv("../data/raw/2022-2023/HMS_2022-2023_PUBLIC_instchars.csv")


In [ ]:
# 2023-2024
hms_24 = pd.read_csv("../data/raw/2023-2024/HMS_2023-2024_PUBLIC_instchars.csv")

In [ ]:
# 2024-2025
hms_25 = pd.read_csv("../data/raw/2024-2025/HMS_2024-2025_PUBLIC_instchars.csv")

In [ ]:
# 2025-2026
hms_26 = pd.read_csv("../data/raw/2025-2026/HMS_2025-2026_PUBLIC_instchars.csv")

In [ ]:
# selected variables

selected_vars = [
    # Outcomes
    "flourish",
    "deprawsc",
    "anx_score",

    # Main factors
    "fincur",
    "finpast",
    "belong1",
    "discrim_race",
    "discrim_culture",
    "discrim_gender",
    "discrim_sexual",
    "alc_any",

    # Controls
    "age",
    "international",
    "educ_par1",
    "educ_par2",
    "enroll",

    # Survey weight
    "nrweight"]

In [ ]:
# create copies of data
hms_23_selected = hms_23[selected_vars].copy()
hms_24_selected = hms_24[selected_vars].copy()
hms_25_selected = hms_25[selected_vars].copy()
hms_26_selected = hms_26[selected_vars].copy()


In [ ]:
# create year columns
hms_23_selected["survey_year"] = "2022-2023"
hms_24_selected["survey_year"] = "2023-2024"
hms_25_selected["survey_year"] = "2024-2025"
hms_26_selected["survey_year"] = "2025-2026"


In [ ]:
# Join the tables together

hms_all = pd.concat([hms_23_selected, hms_24_selected, hms_25_selected, hms_26_selected],
                    ignore_index = True)
hms_all.head()

## Inspect Joined Data

In [ ]:
# shape
hms_all.shape

In [ ]:
# check how many per year
hms_all["survey_year"].value_counts()

In [ ]:
# info
hms_all.info()

In [ ]:
# types
hms_all.dtypes
# will be changing some data types to integer

## Valid Data

In [ ]:
# describe
hms_all.describe()

In [ ]:
# are outcomes in range
hms_all[["flourish", "deprawsc", "anx_score"]].describe().loc[["min", "max"]]

In [ ]:
# are factors in range
hms_all[["fincur", "finpast", "belong1", "discrim_race",
         "discrim_culture", "discrim_gender", "discrim_sexual", "alc_any"]].describe().loc[["min", "max"]]

In [ ]:
# are outcome numbers valid
outcome_cols = ["flourish", "deprawsc", "anx_score"]

for col in outcome_cols:
    print(col, sorted(hms_all[col].dropna().unique()))

In [ ]:
# are factor numbers valid
factor_cols = ["fincur", "finpast", "belong1", "discrim_race",
               "discrim_culture", "discrim_gender", "discrim_sexual","alc_any"]

for col in factor_cols:
    print(col, sorted(hms_all[col].dropna().unique()))

In [ ]:
# are control number valid
control_cols = [
    "age",
    "international",
    "educ_par1",
    "educ_par2",
    "enroll"
]

for col in control_cols:
    print(f"\n{col}")
    print("Min:", hms_all[col].min())
    print("Max:", hms_all[col].max())
    print("Unique:", sorted(hms_all[col].dropna().unique()))

In [ ]:
# Ages that contain a decimal
decimal_ages = hms_all[
    hms_all["age"] % 1 != 0
]

decimal_ages["age"].value_counts().sort_index()

In [ ]:
# ages >= 80
hms_all[hms_all["age"] >= 80]["age"].value_counts().sort_index()

**Variable Validation**: The outcomes, factors, and control variables that were kept were compared with the response coding from he HMS data dictionaries. All outcomes contained only valid whole-number scores within the given range. All factors variables also contained only expected response codings. No invalid coded values were found/identified.

In [ ]:
# what years have decimal ages
decimal_ages.groupby(
    ["survey_year", "age"]
).size()

In [ ]:
# What years have older respondents
hms_all[hms_all["age"] >= 80].groupby(
    ["survey_year", "age"]
).size()

In [ ]:
# how many have age greater than 100
hms_all[hms_all["age"] >= 100].shape[0]
# percent 
hms_all[hms_all["age"] >= 100].shape[0] / len(hms_all) * 100

**Age Validation**: `age` is ranged from 18 to 120 years. Five of the respondents have decimal ages and 10 (approximaely 0.003% of the combined data) have reported ages of 100 or older. The question asked for `age` is *"How old are you?"* and only specifies a minimum age of 18. It does not specify an upper limit or require an integer response. Since these values do not violate the documented coding specified in the HMS codebook and make up a very small proportion of the dataset, age values will be kept without modification.

## Changing Data Types

In [ ]:
# types (again)
hms_all.dtypes

In [ ]:
# changing data types to int nullable except for age, nrweight, and survery year
int_cols = ["flourish", "deprawsc", "anx_score", "fincur",
            "finpast", "belong1", "discrim_race", "discrim_culture",
            "discrim_gender", "discrim_sexual", "alc_any", "international",
            "educ_par1", "educ_par2", "enroll"]

hms_all[int_cols] = hms_all[int_cols].astype("Int64")

In [ ]:
# check types
hms_all.dtypes

**Data Types:** Variables that represen whole-number survey scores or hose coded as categories were converted from `float64` into pandas nullable `Int64`; nullable integers were kept so missing responses could be included. `age` and `nrweight` stayed as `float64` because they have non-integer values. `survey_year` was kept as a `str` text variable.

## Missingness

In [ ]:
# NA values
hms_all.isna().sum()

In [ ]:
# percentage of missingness per column
missing_summary = pd.DataFrame({
    "missing_count": hms_all.isna().sum(),
    "missing_percent": hms_all.isna().mean() * 100
})

missing_summary.sort_values("missing_percent", ascending=False)

In [ ]:
# Missingness grouped by year as percent
hms_all.isna().groupby(hms_all["survey_year"]).mean() * 100

In [ ]:
# Missingness percent for outcomes
hms_all[["flourish", "deprawsc", "anx_score"]].isna().mean() * 100

In [ ]:
# Missingness percent for factors
hms_all[["fincur", "finpast", "belong1", "discrim_race", "discrim_culture", "discrim_gender", "discrim_sexual", "alc_any"]].isna().mean() * 100

**Missingness:** Missingess was assessed for each variable that was kept, both overall and by survey year. Before removing observaions, missingness values ranged from 0% up to approximately 11.5%. Anxiety and depression had he highest levles of missingess, followed by alcohol consumption and flourishing; most factors and controls had lower levels of missingness. Survey years in 2024-2025 and 2025-2026 also had higher missingness. No imputation was done on missing values because the type of missingness cannot be assumed.

### Remove Rows where all Outcomes are Missing

In [ ]:
# remove rows where flourish, deprawsc, and anx_score are missing
hms_all_na_removed = hms_all.dropna(subset = ["flourish", "deprawsc", "anx_score"], how = "all").copy()
hms_all_na_removed.shape

In [ ]:
# see new data NA sum
hms_all_na_removed.isna().sum()

In [ ]:
# compare against old dataset
print(f"Removed: {hms_all.shape[0] - hms_all_na_removed.shape[0]} outcomes")
print(f"Removed: {round(((hms_all.shape[0] - hms_all_na_removed.shape[0]) / hms_all.shape[0] * 100), 3)}% of outcomes")


In [ ]:
# check new dataset with outcome NAs removed when all three were missing
hms_all_na_removed.describe()

In [ ]:
# Missingness percent for outcomes
hms_all_na_removed[["flourish", "deprawsc", "anx_score"]].isna().mean() * 100

In [ ]:
# Missingness percent for factors
hms_all_na_removed[["fincur", "finpast", "belong1", "discrim_race", "discrim_culture", "discrim_gender", "discrim_sexual", "alc_any"]].isna().mean() * 100

In [ ]:
# Missingness grouped by year as percent
hms_all_na_removed.isna().groupby(hms_all_na_removed["survey_year"]).mean() * 100

In [ ]:
missing_summary_na_removed = pd.DataFrame({
    "missing_count": hms_all_na_removed.isna().sum(),
    "missing_percent": hms_all_na_removed.isna().mean() * 100
})

missing_summary_na_removed.sort_values("missing_percent", ascending=False)

**Missingness in Outcomes:** Respondens missing all three outcome variables were removed because they could not properly contribute to any primary outcome analyses. This removed 26474 values (approximately 7.94%) retaining 306873 observations. Respondents who were missing only one or two of he outcomes were kept because they can still contribute to analyses of the other outcomes.

**Missingness in `alc_any`:** `alc_any` responses that were missing were not encoded as `0` or imputed because the response was dependent on if the institution used the Substacne Use module in the survey. Missingness for `alc_any` can be reflected from survey design rather than nonresponse.

## Final Cleaning

In [ ]:
# rename data
hms_clean = hms_all_na_removed.copy()

In [ ]:
# reset indices
hms_clean = hms_clean.reset_index(drop = True)
hms_clean

In [ ]:
# check for duplicates
datasets = {
    "2022-2023": hms_23,
    "2023-2024": hms_24,
    "2024-2025": hms_25,
    "2025-2026": hms_26
}

print("Check for Duplicated Respondents")
for year, df in datasets.items():
    print(year)

    if "responseid" in df.columns:
        print("responseid exists")
        print("Rows:", len(df))
        print("Unique responseid:", df["responseid"].nunique())
        print("Duplicate responseid:", df["responseid"].duplicated().sum())
        print("Missing responseid:", df["responseid"].isna().sum())
    else:
        print("No responseid column")

    print()

In [ ]:
# Final checks
hms_clean.shape
hms_clean.dtypes
hms_clean["survey_year"].value_counts()
hms_clean.columns

In [ ]:
# save clean data
hms_clean.to_csv("../data/processed/hms_clean.csv", index = False)

## Cleaning Summary

The four HMS survey years were combined and validated against the expected variable coding found in the codebooks. Appropriate data types were assigned, unusual values were looked into before modification, and respondents missing all three outcomes were removed. The resulting cleaned dataset contains 306873 respondents and 18 variables. Further analysis will be done using SQL.